<h3 style="font-weight:600;color:#395B91">1 | Delicious Asian and Indian Cuisines</h3>

Working with a dataset of cuisine data, we'll try to predict the *cuisine of origin* by observing the group of *ingredients*.

This is actually a **multiclass question**, since we have several potential national cuisines. Given the batch of ingredients, which of these many classes will the data fit?

<h3 style="font-weight:600;color:#395B91">2 | Clean and Balance your Data</h3>

First we need to install [imblearn](https://imbalanced-learn.org/stable/).

Imbalanced-learn (imported as imblearn) is an open source, MIT-licensed library relying on scikit-learn (imported as sklearn) and provides tools when dealing with classification with **imbalanced classes**.

In [ ]:
# Uncomment and execute the line below to install imblearn (if not already installed).
# !pip install imbalanced-learn

In [ ]:
# Import the required packages
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from imblearn.over_sampling import SMOTE

In [ ]:
# Import the dataset
df = pd.read_csv("../data/cuisines.csv")

In [ ]:
# Check the data's shape
# NOTE: It looks like one-hot encoding has already been applied to the dataset.
#       The columns are the names of the ingredients.
df.head()

In [ ]:
# Show info about the data
df.info()

<h3 style="font-weight:600;color:#395B91">3 | Learning about Cuisines</h3>

Examine the distribution of data per cuisine.

In [ ]:
df["cuisine"].value_counts().plot.barh(color="fuchsia")
plt.show()

> Although we only have 5 unique national cuisines, the distribution of data is **uneven**! We need to fix that.

But first let's explore the data a little more. What is the data frequency per cuisine?

In [ ]:
thai_df = df[df.cuisine == "thai"]
japanese_df = df[df.cuisine == "japanese"]
chinese_df = df[df.cuisine == "chinese"]
indian_df = df[df.cuisine == "indian"]
korean_df = df[df.cuisine == "korean"]

print("-" * 30)
print("Data Distribution per Cuisine")
print("-" * 30)
print(f"thai df:\t{thai_df.shape[0]}")
print(f"japanese df:\t{japanese_df.shape[0]}")
print(f"chinese df:\t{chinese_df.shape[0]}")
print(f"indian df:\t{indian_df.shape[0]}")
print(f"korean df:\t{korean_df.shape[0]}")

<h3 style="font-weight:600;color:#395B91">4 | Discovering Ingredients</h3>

<h4 style="font-weight:600;color:#5B7DB1">4.1 | Create function to return the frequency of each ingredient</h4>

In [ ]:
def create_ingredient_df(df: pd.DataFrame):
    # Return a df with the ingredients as index labels and the count of each as value:
    ingredient_df = df.T.drop(["cuisine", "Unnamed: 0"]).sum(axis=1).to_frame("value")
    # Drop columns where the specific ingredient is not used
    ingredient_df = ingredient_df[(ingredient_df.T != 0).any()]
    # Sort through the ingredients by their count:
    ingredient_df = ingredient_df.sort_values(by="value", ascending=False, inplace=False)

    return ingredient_df

<h4 style="font-weight:600;color:#5B7DB1">4.2 | Use the function to get the 10 most popular ingredients per Cuisine</h4>

<h5 style="font-weight:600;color:#6C8EC5">4.2.1 | Thai Cuisine</h5>

In [ ]:
thai_ingredient_df = create_ingredient_df(thai_df)
thai_ingredient_df.head(10).plot.barh()
plt.show()

<h5 style="font-weight:600;color:#6C8EC5">4.2.2 | Japanese Cuisine</h5>

In [ ]:
japanese_ingredient_df = create_ingredient_df(japanese_df)
japanese_ingredient_df.head(10).plot.barh()
plt.show()

<h5 style="font-weight:600;color:#6C8EC5">4.2.3 | Chinese Cuisine</h5>

In [ ]:
chinese_ingredients_df = create_ingredient_df(chinese_df)
chinese_ingredients_df.head(10).plot.barh()
plt.show()

<h5 style="font-weight:600;color:#6C8EC5">4.2.4 | Indian Cuisine</h5>

In [ ]:
indian_ingredients_df = create_ingredient_df(indian_df)
indian_ingredients_df.head(10).plot.barh()
plt.show()

<h5 style="font-weight:600;color:#6C8EC5">4.2.5 | Korean Cuisine</h5>

In [ ]:
korean_ingredients_df = create_ingredient_df(korean_df)
korean_ingredients_df.head(10).plot.barh()
plt.show()

For a more compact presentation, try the following:

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(10, 10), layout="constrained")

cuisine_title = ["Thai Cuisine", "Japanese Cuisine", "Chinese Cuisine", "Indian Cuisine", "Korean Cuisine"]
cuisine_color = ["red", "green", "brown", "blue", "orange"]
cuisine_df = [thai_df, japanese_df, chinese_df, indian_df, korean_df]

for i in range(len(cuisine_df)):
    ingredients_df = create_ingredient_df(cuisine_df[i]).head(10)
    ax = axs.flat[i]
    ax.barh(ingredients_df.index, ingredients_df.value, color=cuisine_color[i])
    ax.set_title(cuisine_title[i])

plt.show()

Drop the most common ingredients that create confusion between distinct cuisines.<br/>
Everyone loves rice, garlic and ginger!

In [ ]:
feature_df = df.drop(["cuisine", "Unnamed: 0", "rice", "garlic", "ginger"], axis=1)
labels_df = df.cuisine  # .unique()
feature_df.head()

<h3 style="font-weight:600;color:#395B91">5 | Balance the Dataset</h3>

Use [SMOTE](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html) - Synthetic Minority Over-sampling Technique to balance the dataset.

In [ ]:
# Generate new samples by interpolation, by calling fit_resample()
oversample = SMOTE()
transformed_features_df, transformed_label_df = oversample.fit_resample(feature_df, labels_df)

In [ ]:
# Check the number of labels per ingredient
print(f"New label count: {transformed_label_df.value_counts()}")
print(f"Old label count: {df.cuisine.value_counts()}")

Now the data is nice and clean and balanced!

Let's save the balanced data (including labels and features) into a new dataframe that can be exported into a file:

In [ ]:
transformed_df = pd.concat([transformed_label_df, transformed_features_df], axis=1, join="outer")

Display the new transformed data:

In [ ]:
transformed_df.head()

In [ ]:
transformed_df.info()

Save a copy of this data:

In [ ]:
transformed_df.to_csv("../data/cleaned_cuisines.csv")